# RAG_01: Transcript Indexing & Vector DB Construction

Parse raw transcripts, segment into conversational **states** (decision points
after each customer utterance), extract multi-scale context windows, generate
GPT-4.1 structured summaries, and embed everything with ada-2.

**Input:** `transcripts_df` — a pandas DataFrame you load yourself (see stub cell below).
Required columns: `conversation_id`, `transcript`. Optional: `label` (conversion outcome).

**Output:**
- `data/rag_embeddings.npz` — 6 numpy arrays of shape `(N_states, 1536)`, one per window scale
- `data/rag_metadata.parquet` — N_states rows with window texts, agent responses, and metadata
- `data/rag_summaries_cache.jsonl` — cached GPT-4.1 summaries (crash-safe, incremental)

In [ ]:
import re
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

# ============================================================
# Configuration
# ============================================================
OUTPUT_DIR = Path("data")
EMBEDDINGS_FILE = OUTPUT_DIR / "rag_embeddings.npz"
METADATA_FILE = OUTPUT_DIR / "rag_metadata.parquet"
SUMMARIES_CACHE_FILE = OUTPUT_DIR / "rag_summaries_cache.jsonl"

EMBEDDING_DIM = 1536  # ada-2
WINDOW_NAMES = ['w1', 'w2', 'w3', 'w5', 'w10', 'w_summary']

# Regex for parsing transcripts (same as notebook 01)
TURN_PATTERN = re.compile(r'^(agent|customer)\(u(\d+)\):\s*(.+)$', re.MULTILINE)

## Load Data (placeholder — fill in your own loading logic)

The cell below is a stub. Replace it with your actual data loading code.
The result must be a DataFrame called `transcripts_df` with at least:
- `conversation_id` (str): unique identifier per conversation
- `transcript` (str): raw transcript in `agent(u1): ... \n customer(u2): ...` format

Optional column: `label` (int, 1 = converted, 0 = not converted)

In [ ]:
# TODO: Replace with your actual data loading logic
# Examples:
#   transcripts_df = pd.read_csv("data/transcripts.csv")
#   transcripts_df = pd.read_json("data/raw_transcripts.jsonl", lines=True)

transcripts_df = ...  # <-- FILL IN

# Validate schema
assert isinstance(transcripts_df, pd.DataFrame), "transcripts_df must be a pandas DataFrame"
assert 'conversation_id' in transcripts_df.columns, "Missing column: conversation_id"
assert 'transcript' in transcripts_df.columns, "Missing column: transcript"

print(f"Loaded {len(transcripts_df)} transcripts")
print(f"Columns: {list(transcripts_df.columns)}")
transcripts_df.head(2)

## Load Embedding & Summarization Models

In [ ]:
from safechain.lcel import model

embedder = model('ada-2')

# GPT-4.1 for structured summaries.
# If safechain doesn't expose GPT-4.1 as a chat model, replace with a direct
# Azure OpenAI SDK call.
summarizer = model('gpt-4.1')

# Quick sanity check
test_vec = embedder.embed_query("hello world")
print(f"Embedding dim: {len(test_vec)} (expected {EMBEDDING_DIM})")
assert len(test_vec) == EMBEDDING_DIM

## Parse Transcripts & Identify States

In [ ]:
def parse_transcript(raw_text: str) -> list[dict]:
    """Parse 'agent(u1): ...' format into list of {'speaker', 'index', 'text'}."""
    matches = TURN_PATTERN.findall(raw_text)
    return [
        {"speaker": s, "index": int(i), "text": t.strip()}
        for s, i, t in matches
    ]


def identify_states(turns: list[dict]) -> list[int]:
    """Return list-indices of customer utterances (= state boundaries).

    Each state is a decision point: the agent must choose what to say next.
    We only keep states where a subsequent agent response exists (so we can
    store what the agent actually said).
    """
    states = []
    for i, turn in enumerate(turns):
        if turn["speaker"] == "customer":
            # Check that there's at least one more turn (the agent response)
            if i + 1 < len(turns):
                states.append(i)
    return states


# Quick test
sample_turns = parse_transcript(transcripts_df.iloc[0]["transcript"])
sample_states = identify_states(sample_turns)
print(f"Sample conversation: {len(sample_turns)} turns, {len(sample_states)} states")
print(f"First 3 state indices: {sample_states[:3]}")

## Extract Multi-Scale Windows

In [ ]:
def _turns_to_text(turns: list[dict]) -> str:
    """Concatenate turns into a single string using special-token markers."""
    lines = []
    for t in turns:
        marker = "<|agent|>" if t["speaker"] == "agent" else "<|customer|>"
        lines.append(f"{marker}{t['text']}")
    return "\n".join(lines)


def extract_windows(turns: list[dict], state_turn_idx: int) -> dict[str, str]:
    """Extract 5 text windows (w1, w2, w3, w5, w10) for the state at state_turn_idx.

    state_turn_idx is the index of the customer utterance defining this state.
    Windows look back from (and include) that utterance.
    If fewer turns are available than the window size, all available turns are used.
    """
    # All turns up to and including the customer utterance at state_turn_idx
    available = turns[:state_turn_idx + 1]

    windows = {}
    windows['w1'] = turns[state_turn_idx]['text']  # just the customer utterance text
    windows['w2'] = _turns_to_text(available[-2:])
    windows['w3'] = _turns_to_text(available[-3:])
    windows['w5'] = _turns_to_text(available[-5:])
    windows['w10'] = _turns_to_text(available[-10:])

    return windows


# Quick test
sample_windows = extract_windows(sample_turns, sample_states[min(2, len(sample_states)-1)])
for name, text in sample_windows.items():
    print(f"{name} ({len(text)} chars): {text[:80]}...")
    print()

## Structured Summary Generation (w_summary) with Caching

In [ ]:
SUMMARY_PROMPT = """Summarize the following sales call context in this exact format:
- Conversation phase: (opening / discovery / pitch / objection_handling / closing / follow_up)
- Treatment: (e.g., AP automation, employee cards, cross-sell, underwriting, etc.)
- Customer intent: (interested / skeptical / comparing_alternatives / ready_to_commit / disengaged / unclear)
- Key objection or topic: (one sentence)
- Sentiment trajectory: (warming / cooling / neutral / volatile)
- Customer type signals: (any inferred seniority, role, company size indicators)

Conversation so far:
{conversation_text}"""


def load_summary_cache(cache_path: Path) -> dict[str, str]:
    """Load existing summaries from JSONL cache. Key = 'conv_id|state_idx'."""
    cache = {}
    if cache_path.exists():
        with open(cache_path, 'r', encoding='utf-8') as f:
            for line in f:
                entry = json.loads(line)
                cache[entry['key']] = entry['summary']
    return cache


def save_summary_to_cache(cache_path: Path, key: str, summary: str):
    """Append one summary to the cache file (incremental, crash-safe)."""
    with open(cache_path, 'a', encoding='utf-8') as f:
        f.write(json.dumps({'key': key, 'summary': summary}, ensure_ascii=False) + '\n')


def generate_summary(
    turns: list[dict],
    state_turn_idx: int,
    conv_id: str,
    state_index: int,
    cache: dict[str, str],
    cache_path: Path,
) -> str:
    """Generate a structured summary via GPT-4.1, with disk-backed cache."""
    key = f"{conv_id}|{state_index}"
    if key in cache:
        return cache[key]

    # Build full conversation text up to this state
    available = turns[:state_turn_idx + 1]
    conversation_text = _turns_to_text(available)
    prompt = SUMMARY_PROMPT.format(conversation_text=conversation_text)

    summary = summarizer.invoke(prompt)
    if hasattr(summary, 'content'):
        summary = summary.content  # handle ChatModel response objects

    cache[key] = summary
    save_summary_to_cache(cache_path, key, summary)
    return summary


# Load existing cache (empty on first run)
summary_cache = load_summary_cache(SUMMARIES_CACHE_FILE)
print(f"Summary cache: {len(summary_cache)} entries loaded")

## Embedding Functions

In [ ]:
def embed_text(text: str) -> np.ndarray:
    """Embed a single text string using ada-2, return L2-normalized 1536-dim vector."""
    vec = np.array(embedder.embed_query(text), dtype='float32')
    vec /= (np.linalg.norm(vec) + 1e-8)
    return vec


def embed_state_windows(window_texts: dict[str, str]) -> dict[str, np.ndarray]:
    """Embed all 6 windows for one state (w1..w10 + w_summary).

    Returns dict of window_name -> L2-normalized vector.
    """
    return {name: embed_text(window_texts[name]) for name in WINDOW_NAMES}

## Main Indexing Loop

For each transcript: parse turns → identify states → for each state: extract
windows, generate GPT-4.1 summary, embed all 6 windows, collect metadata.

In [ ]:
# Accumulators
vectors = {name: [] for name in WINDOW_NAMES}
metadata_rows = []

for _, row in tqdm(transcripts_df.iterrows(), total=len(transcripts_df), desc="Transcripts"):
    conv_id = str(row['conversation_id'])
    turns = parse_transcript(row['transcript'])
    label = row.get('label', None)

    if len(turns) < 2:
        continue

    state_indices = identify_states(turns)

    for state_num, state_turn_idx in enumerate(state_indices):
        # --- Extract text windows (w1..w10) ---
        window_texts = extract_windows(turns, state_turn_idx)

        # --- Generate structured summary (w_summary) ---
        summary = generate_summary(
            turns, state_turn_idx, conv_id, state_num,
            summary_cache, SUMMARIES_CACHE_FILE,
        )
        window_texts['w_summary'] = summary

        # --- Embed all 6 windows ---
        window_vecs = embed_state_windows(window_texts)
        for name in WINDOW_NAMES:
            vectors[name].append(window_vecs[name])

        # --- Find the agent response that follows this state ---
        agent_response = turns[state_turn_idx + 1]['text']
        agent_response_turn_idx = turns[state_turn_idx + 1]['index']

        # --- Collect metadata ---
        metadata_rows.append({
            'conversation_id': conv_id,
            'state_index': state_num,
            'state_turn_idx': state_turn_idx,
            'agent_response': agent_response,
            'agent_response_turn_idx': agent_response_turn_idx,
            'w1_text': window_texts['w1'],
            'w2_text': window_texts['w2'],
            'w3_text': window_texts['w3'],
            'w5_text': window_texts['w5'],
            'w10_text': window_texts['w10'],
            'w_summary_text': window_texts['w_summary'],
            'n_turns_available': state_turn_idx + 1,
            'label': label,
        })

metadata_df = pd.DataFrame(metadata_rows)
print(f"\nIndexed {len(metadata_df)} states from {transcripts_df['conversation_id'].nunique()} conversations")

## Validation & Statistics

In [ ]:
# Stack vectors into matrices
embedding_store = {name: np.vstack(vectors[name]) for name in WINDOW_NAMES}

# Validate shapes
for name in WINDOW_NAMES:
    shape = embedding_store[name].shape
    print(f"  {name}: {shape}")
    assert shape == (len(metadata_df), EMBEDDING_DIM), f"Shape mismatch for {name}"

print(f"\nTotal states: {len(metadata_df)}")
print(f"Conversations: {metadata_df['conversation_id'].nunique()}")
print(f"\nStates per conversation:")
print(metadata_df.groupby('conversation_id').size().describe())
print(f"\nTurns available at state time (n_turns_available):")
print(metadata_df['n_turns_available'].describe())
print(f"\nSample metadata rows:")
metadata_df[['conversation_id', 'state_index', 'n_turns_available', 'agent_response']].head(3)

## Save Embeddings & Metadata

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)

np.savez(str(EMBEDDINGS_FILE), **embedding_store)
metadata_df.to_parquet(str(METADATA_FILE), index=False)

print(f"Saved embeddings to {EMBEDDINGS_FILE}")
print(f"Saved metadata  to {METADATA_FILE}")
print(f"File sizes:")
print(f"  embeddings: {EMBEDDINGS_FILE.stat().st_size / 1e6:.1f} MB")
print(f"  metadata:   {METADATA_FILE.stat().st_size / 1e6:.1f} MB")

## Smoke Test: Quick Retrieval Sanity Check

Pick one state and find its top-5 matches using uniform weights. This is just
a quick visual check — formal retrieval and weight optimization happen in
RAG_02 and RAG_03.

In [ ]:
# Pick a state from the middle of a conversation
query_idx = len(metadata_df) // 2
query_row = metadata_df.iloc[query_idx]
query_conv_id = query_row['conversation_id']

print(f"Query state: conversation={query_conv_id}, state_index={query_row['state_index']}")
print(f"Customer said (w1): {query_row['w1_text'][:200]}")
print(f"Agent responded:    {query_row['agent_response'][:200]}")
print()

# Compute composite scores with uniform weights
uniform_weights = {name: 1.0 / len(WINDOW_NAMES) for name in WINDOW_NAMES}
n_states = embedding_store['w1'].shape[0]
composite_scores = np.zeros(n_states, dtype='float32')

for name in WINDOW_NAMES:
    q = embedding_store[name][query_idx]
    sims = embedding_store[name] @ q
    composite_scores += uniform_weights[name] * sims

# Exclude same conversation
mask = (metadata_df['conversation_id'] == query_conv_id).values
composite_scores[mask] = -np.inf

# Top 5
top5 = np.argsort(composite_scores)[::-1][:5]

print("Top 5 matches:")
print("-" * 80)
for rank, idx in enumerate(top5, 1):
    r = metadata_df.iloc[idx]
    print(f"\n#{rank} (score={composite_scores[idx]:.4f}) "
          f"conv={r['conversation_id']}, state={r['state_index']}")
    print(f"  Customer: {r['w1_text'][:150]}")
    print(f"  Agent:    {r['agent_response'][:150]}")